# SBCross

In [ ]:
#Version: v1.0

# Changelog
# Open version v1.0 published 20.08.2026 by Stephan G. Heinemann

## Load Packages and Functions

In [8]:
# Standard library imports
import copy
import dataclasses
import datetime
from datetime import datetime as dtime, timedelta
import glob
import importlib
import os
import pickle
import re
import sys
import time

# Third-party library imports
from astropy.constants import au
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.time import Time, TimeDelta
import astropy.units as u
from astropy.visualization import time_support
from astropy.wcs import WCS
from cdasws import CdasWs
import csv
import matplotlib.dates as mdates
import matplotlib.lines as mlines
from matplotlib.dates import date2num, num2date
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numba
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr
from skimage.measure import find_contours
import sunpy
from sunpy.coordinates import HeliographicCarrington, HeliographicStonyhurst
from sunpy.coordinates.sun import carrington_rotation_number, carrington_rotation_time
import sunpy.map
from sunpy.map.header_helper import make_heliographic_header
from sunpy.net import Fido, attrs as a
import sunpy.timeseries as ts
from sunpy.time import TimeRange


def is_in_or_near_interval(time_val: float, start_times: list[float], end_times: list[float]) -> bool:
    """
    Check if a given matplotlib numerical time value falls within or within 
    +/- 1 day of any defined (start_time, end_time) intervals.

    Parameters
    ----------
    time_val : float
        Target time expressed in matplotlib date format (date2num).
    start_times : list of float
        Interval start times in date2num format.
    end_times : list of float
        Interval end times in date2num format.

    Returns
    -------
    bool
        True if target time is within or near (+/- 1 day) any interval.
    """
    buffer_one_day = 1.0  # 1.0 unit corresponds to 1 day in matplotlib's date2num format

    for start, end in zip(start_times, end_times):
        # Expand interval by +/- 1 day
        if (start - buffer_one_day) <= time_val <= (end + buffer_one_day):
            return True

    return False

In [78]:
def gaussian_smooth_with_nans(data, sigma):
    """
    Applies Gaussian smoothing to a 1D array or Pandas Series while ignoring NaN values.
    Uses normalized convolution to prevent NaNs from contaminating adjacent data.
    
    Parameters:
    -----------
    data : array-like
        The 1D array or Pandas Series containing the data to smooth (can include NaNs).
    sigma : float
        Standard deviation of the Gaussian kernel (in index/pixel units, e.g., hours).
        
    Returns:
    --------
    np.ndarray
        The smoothed 1D array with NaN gaps preserved.
    """
    # Ensure input is a numpy array copy so we don't accidentally mutate original data
    y = np.asarray(data, dtype=float).copy()
    
    # Create a mask: 1.0 for valid data, 0.0 for NaNs
    mask = np.isfinite(y).astype(float)
    
    # Temporarily zero-out NaNs so they don't break the convolution math
    y[~np.isfinite(y)] = 0.0
    
    # Smooth both the data and the mask
    smoothed_y = gaussian_filter1d(y, sigma=sigma, mode='constant', cval=0.0)
    smoothed_mask = gaussian_filter1d(mask, sigma=sigma, mode='constant', cval=0.0)
    
    # Normalize by dividing smoothed data by smoothed mask weight.
    # Where mask weight is 0 (complete data void), we return NaN.
    with np.errstate(divide='ignore', invalid='ignore'):
        smoothed_data = np.where(smoothed_mask > 0, smoothed_y / smoothed_mask, np.nan)
        
    return smoothed_data
    

def gaussian_smooth_angles(angles, sigma):
    x = np.cos(angles)
    y = np.sin(angles)
    
    x_smooth = gaussian_filter1d(x, sigma=sigma)
    y_smooth = gaussian_filter1d(y, sigma=sigma)
    
    return np.arctan2(y_smooth, x_smooth)
    

def process_and_smooth_azimuth(bx, by, dates, sigma=12, nosmooth=False):
    # 1. Standardize dates
    dates_pd = pd.to_datetime(dates)
    
    # 2. Fix NaT dates using Series interpolation directly on the datetime objects
    # This avoids the 'nearly the same date' bug caused by manual float conversion
    if dates_pd.isna().any():
        # We use 'linear' on the series; pandas handles the datetime spacing correctly
        dates_pd = pd.Series(dates_pd).interpolate(method='linear').ffill().bfill()

    # 3. Create DataFrame - Use a fresh RangeIndex to avoid duplicate-index errors
    df = pd.DataFrame({
        'bx': np.asanyarray(bx), 
        'by': np.asanyarray(by), 
        'dt': dates_pd.values
    })
    
    # 4. Cartesian Components
    mag = np.sqrt(df['bx']**2 + df['by']**2)
    df['x'] = df['bx'] / mag
    df['y'] = df['by'] / mag

    # 5. Gap Logic (> 12 hours)
    is_nan = df['x'].isna()
    # Identify contiguous blocks
    nan_groups = (is_nan != is_nan.shift()).cumsum()
    
    # Calculate duration of each block using the 'dt' column
    # transform ensures the result is the same length as the original dataframe
    group_durations = df.groupby(nan_groups)['dt'].transform(lambda x: x.iloc[-1] - x.iloc[0])
    long_gap_mask = is_nan & (group_durations > pd.Timedelta(hours=12))

    # 6. Interpolate Unit Vectors (Time-based)
    # To interpolate by time, we temporarily set the index to 'dt'
    temp_df = df[['dt', 'x', 'y']].set_index('dt')
    # Use 'time' method to respect actual temporal spacing
    df['x_interp'] = temp_df['x'].interpolate(method='time').values
    df['y_interp'] = temp_df['y'].interpolate(method='time').values

    # 7. Smoothing
    if nosmooth or sigma == 0:
        x_smooth, y_smooth = df['x_interp'], df['y_interp']
    else:
        # Fill remaining NaNs (only at edges/long gaps) for Gaussian kernel
        x_filled = df['x_interp'].ffill().bfill()
        y_filled = df['y_interp'].ffill().bfill()
        
        if x_filled.isna().all():
            x_smooth, y_smooth = x_filled, y_filled
        else:
            x_smooth = gaussian_filter1d(x_filled, sigma=sigma)
            y_smooth = gaussian_filter1d(y_filled, sigma=sigma)

    # 8. Reconstruct Azimuth
    processed_az = np.arctan2(y_smooth, x_smooth)

    # 9. Apply Final Mask
    # Mask if it was a gap > 12h or if it stayed NaN (start/end of data)
    final_mask = long_gap_mask | df['x_interp'].isna()
    final_az = np.where(final_mask, np.nan, processed_az)

    # Return dates as numpy datetime64[ns]
    return final_az, df['dt'].to_numpy()

def normalised_pad(energy_channel):
    
    nan2be = np.min(energy_channel)

    energy_channel = np.where(energy_channel==nan2be, np.nan, energy_channel)
    np.nanmax(energy_channel)

    
    energy_channel_means = np.nanmean(energy_channel, axis=0) #column means

    energy_channel_normalized = energy_channel / energy_channel_means
    
    return(energy_channel_normalized)

## Set Parameters

In [100]:
# Set path for output
path='/Users/sgh/Documents/Projects/HCS_statistics/'

# Set path to RC_list, set to None if no list available 
#dson	ICME Plasma/Field Start, End Y/M/D (UT) (b) 		Comp. Start, End (Hrs wrt. Plasma/ Field) (c) 		MC Start, End (Hrs wrt. Plasma/ Field) (d) 		BDE? (e) 	BIF? (f) 	Qual. (g)	dV (km/s) (h) 	V_ICME (km/s) (i) 	V_max (km/s) (j) 	B (nT) (k) 	MC? (l) 	Dst (nT) (m) 	V_transit (km/s) (n) 	LASCO CME Y/M/D (UT) (o)
#1996/05/27 1500	1996/05/27 1500	1996/05/29 0300	... 	... 	0	4	N 	... 	2	0	370	400	9	2	-33	... 	 

CME_list = path+'RC_list.csv'


# Set start and end time
start = "2023-01-01T00:00:00Z"
end = "2023-04-10T00:00:00Z"

# Detection Setup (dont change for standard configuration)
smooth = 6
ds = 2.
min_len = 0
stability_hours = ds * 24
stability_td = np.timedelta64(int(stability_hours), 'h')

## Load Data

In [75]:
# =============================================================================
# 1. CME DATA LOADING
# =============================================================================
rc_start = []
rc_end = []

if CME_list is not None:    
    # Parse CME catalog file
    with open(CME_list, mode="r", encoding="utf-8") as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=",")
        line_count = 0
        for row in csv_reader:
            if line_count == 0:
                line_count += 1
                continue
    
            st = row[0]
            ed = row[2]
    
            # Extract start datetime: YYYY-MM-DD HH:MM
            rc_start.append(
                dtime(
                    int(st[:4]),
                    int(st[5:7]),
                    int(st[8:10]),
                    int(st[11:13]),
                    int(st[13:15]),
                )
            )
            # Extract end datetime: YYYY-MM-DD HH:MM
            rc_end.append(
                dtime(
                    int(ed[:4]),
                    int(ed[5:7]),
                    int(ed[8:10]),
                    int(ed[11:13]),
                    int(ed[13:15]),
                )
            )
            line_count += 1
    
        print(f"Processed {line_count} lines.")
    


# =============================================================================
# 2. OMNI & PAD DATA RETRIEVAL FUNCTIONS
# =============================================================================

def get_omni_hourly_data(
    start_time: str,
    end_time: str,
    variables: list[str] | None = None,
    dataset: str = "OMNI2_H0_MRG1HR",
) -> pd.DataFrame:
    """Fetch 1-hour OMNI solar wind data from NASA CDAWeb without MultiIndex duplication."""
    if variables is None:
        variables = [
            "BX_GSE1800",
            "BY_GSE1800",
            "BZ_GSE1800",
            "T1800",
            "N1800",
            "V1800",
            "Beta1800",
        ]

    cdas = CdasWs()
    print(f"Downloading {dataset} from NASA CDAS...")
    status, data = cdas.get_data(dataset, variables, start_time, end_time)

    if data is None:
        raise ValueError(f"Failed to retrieve data from CDAS. Status: {status}")

    df = data.to_dataframe()

    # Fix MultiIndex duplication: keep only Epoch_1800 and remove cross-product rows
    if isinstance(df.index, pd.MultiIndex):
        if "Epoch" in df.index.names:
            df = df.droplevel("Epoch")

    df = df[~df.index.duplicated(keep="first")]

    # Clean fill values (> 90000) using vectorized masking
    df = df.mask(df > 90000)

    return df


import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d


def fetch_and_process_pad_asym255(
    start_time: str = "2023-01-01T00:00:00Z",
    end_time: str = "2023-01-10T00:00:00Z",
    dataset: str = "WI_ELPD_3DP",
    sigma_hours: float = 6.0,
    target_index: pd.DatetimeIndex | None = None,
) -> pd.DataFrame:
    """Fetch Wind 3DP PAD data, compute asym_255, bin e255 PAD bins, and align to target timeline."""
    cdas = CdasWs()

    print(f"Downloading {dataset} from NASA CDAS...")
    status, data = cdas.get_data(dataset, ["FLUX"], start_time, end_time)

    if data is None or "FLUX" not in data:
        raise ValueError(f"Failed to load data. CDAS Status: {status}")

    flux_xr = data["FLUX"]
    raw_vals = flux_xr.values.astype(float)

    # 1. Target 255 eV channel (index 4)
    target_e_idx = 4
    if raw_vals.ndim == 3:
        if raw_vals.shape[1] == 15:
            e255_slice = raw_vals[:, target_e_idx, :]
        else:
            e255_slice = raw_vals[:, :, target_e_idx]
    else:
        raise ValueError(f"Unexpected FLUX dimensions: {raw_vals.shape}")

    # 2. Filter fill values
    e255_slice[e255_slice <= 0] = np.nan
    e255_slice[e255_slice > 1e20] = np.nan

    # 3. Normalize channel across pitch angles per time step
    with np.errstate(divide="ignore", invalid="ignore"):
        channel_means = np.nanmean(e255_slice, axis=1, keepdims=True)
        norm_slice = e255_slice / channel_means

        flux_par = norm_slice[:, 0]
        flux_ant = norm_slice[:, -1]

        asym_255 = (flux_par - flux_ant) / (flux_par + flux_ant)

    # 4. FIX TIME ALIGNMENT: Strip timezones
    raw_time = pd.to_datetime(data["Epoch"].values)
    if raw_time.tz is not None:
        raw_time = raw_time.tz_convert("UTC").tz_localize(None)

    # Create raw DataFrame containing BOTH asym_255 and individual pitch angle bins
    num_pa_bins = e255_slice.shape[1]
    pa_col_names = [f"pa_{i}" for i in range(num_pa_bins)]

    # Store raw un-normalized or normalized 255 eV flux matrix along with asymmetry
    df_raw = pd.DataFrame(e255_slice, index=raw_time, columns=pa_col_names)
    df_raw["asym_255"] = asym_255

    # 5. Bin raw PAD & asymmetry data to match OMNI's timeline cadence
    if target_index is not None:
        minute_offset = target_index[0].minute
        group_time = df_raw.index.floor("h") + pd.Timedelta(
            minutes=minute_offset
        )
        df_hourly = df_raw.groupby(group_time).mean()
        df_final = df_hourly.reindex(target_index)
    else:
        group_time = df_raw.index.floor("h")
        df_hourly = df_raw.groupby(group_time).mean()
        start_dt = pd.to_datetime(start_time).tz_localize(None).floor("h")
        end_dt = pd.to_datetime(end_time).tz_localize(None).floor("h")
        complete_timeline = pd.date_range(
            start=start_dt, end=end_dt, freq="1h"
        )
        df_final = df_hourly.reindex(complete_timeline)

    df_final.index.name = "Time"

    # 6. NaN-weighted Gaussian Smoothing on asym_255
    y = df_final["asym_255"].values.copy()
    mask = np.isfinite(y).astype(float)
    y[~np.isfinite(y)] = 0.0

    smoothed_y = gaussian_filter1d(
        y, sigma=sigma_hours, mode="constant", cval=0.0
    )
    smoothed_mask = gaussian_filter1d(
        mask, sigma=sigma_hours, mode="constant", cval=0.0
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        y_gaussian = np.where(
            smoothed_mask > 0, smoothed_y / smoothed_mask, np.nan
        )

    df_final["asym_255_gaussian"] = y_gaussian

    # Extract pitch angle metadata (if available) for downstream plotting
    if "PA" in data:
        df_final.attrs["pitch_angles"] = data["PA"].values.astype(float)

    return df_final


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":

    # 1. Fetch OMNI data first
    df_omni = get_omni_hourly_data(start, end)

    # 2. Fetch PAD data binned explicitly to OMNI's index
    df_pad = fetch_and_process_pad_asym255(
        start_time=start,
        end_time=end,
        sigma_hours=6.0,
        target_index=df_omni.index,
    )

    print("Downloading Done ...")

Processed 620 lines.


/var/folders/bq/gbxqd31127n61zblp_qxplbh0000gn/T/ipykernel_1796/4027199048.py:129: RuntimeWarning: Mean of empty slice
  channel_means = np.nanmean(e255_slice, axis=1, keepdims=True)


## Sector Boundary Detection

In [41]:
# =============================================================================
# IMF AZIMUTH & SECTOR TRANSITION IDENTIFICATION
# =============================================================================

# Define target and buffer quadrant IDs
target_ids = [0, 2]
buffer_ids = [1, 3]

# --- 1. Identify Blocks (With Explicit Gap Detection) ---
# Extract inputs directly from df_omni
bx = df_omni["BX_GSE1800"].values
by = df_omni["BY_GSE1800"].values

# FIXED: Handle MultiIndex vs SingleIndex to safely extract datetime values
if isinstance(df_omni.index, pd.MultiIndex):
    # Extract the level containing timestamps (matches 'Time', 'DATETIME', or defaults to level 0)
    time_level = next(
        (name for name in df_omni.index.names if name and "time" in name.lower()), 0
    )
    time_values = df_omni.index.get_level_values(time_level)
else:
    time_values = df_omni.index

az_dates = pd.to_datetime(time_values).to_numpy(dtype="datetime64[ns]")

# Calculate smoothed azimuth angles and timestamps
az_full, az_dates = process_and_smooth_azimuth(bx, by, az_dates, sigma=smooth)

# Bin azimuth angles into 4 directional sectors/quadrants
bins = [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi]
full_segment_indices = np.digitize(az_full, bins) - 1
full_segment_indices[full_segment_indices == 4] = 3
is_valid = np.isfinite(az_full)

blocks_raw = []
if len(az_full) > 0:
    curr_idx = 0
    curr_seg = full_segment_indices[curr_idx] if is_valid[curr_idx] else -1
    block_start_idx = curr_idx

    for i in range(1, len(full_segment_indices)):
        # Check for gaps greater than 1 hour
        time_gap_hours = (az_dates[i] - az_dates[i - 1]) / np.timedelta64(1, "h")
        is_gap = time_gap_hours > 1.0
        this_seg = full_segment_indices[i] if is_valid[i] else -1

        if this_seg != curr_seg or is_gap:
            dur = int((az_dates[i - 1] - az_dates[block_start_idx]) / np.timedelta64(1, "h")) + 1
            blocks_raw.append(
                {
                    "seg": curr_seg,
                    "dur": dur,
                    "start_date": az_dates[block_start_idx],
                    "end_date": az_dates[i - 1],
                    "start_idx": block_start_idx,
                }
            )
            curr_seg = this_seg
            block_start_idx = i

    # Final block append
    dur = int((az_dates[-1] - az_dates[block_start_idx]) / np.timedelta64(1, "h")) + 1
    blocks_raw.append(
        {
            "seg": curr_seg,
            "dur": dur,
            "start_date": az_dates[block_start_idx],
            "end_date": az_dates[-1],
            "start_idx": block_start_idx,
        }
    )

# --- 1b. Filter NaNs and Merge Adjacent Identical Segments ---
blocks = []
for b in blocks_raw:
    if b["seg"] == -1:
        continue
    if len(blocks) > 0 and blocks[-1]["seg"] == b["seg"]:
        blocks[-1]["dur"] += b["dur"]
        blocks[-1]["end_date"] = b["end_date"]
    else:
        blocks.append(b.copy())


# --- Step 2: Extract Candidate Transitions (Buffer-to-Buffer) ---
candidate_transitions = []

for i in range(len(blocks)):
    if blocks[i]["seg"] not in buffer_ids:
        continue

    origin_seg = blocks[i]["seg"]

    # Forward scan to locate destination buffer segment
    for j in range(i + 1, len(blocks)):
        if blocks[j]["seg"] in buffer_ids:
            dest_seg = blocks[j]["seg"]

            # Opposite buffer hit indicates a transition candidate
            if dest_seg != origin_seg:
                t_start = blocks[i]["end_date"]
                t_end = blocks[j]["start_date"]

                duration_hours = (t_end - t_start) / np.timedelta64(1, "h")
                if duration_hours >= min_len:
                    candidate_transitions.append(
                        {
                            "t_start": t_start,
                            "t_end": t_end,
                            "polarity": (origin_seg, dest_seg),
                        }
                    )

            # Any buffer hit halts forward search to reset origin
            break


# --- Step 3: Vectorized Guard-Window Filtering ---
signal_validated_candidates = []

for candidate in candidate_transitions:
    t_start = candidate["t_start"]
    t_end = candidate["t_end"]
    origin_seg, dest_seg = candidate["polarity"]

    # 1. Check Guard Window BEFORE Transition (Lookback)
    t_lookback = t_start - stability_td
    lookback_idx = np.searchsorted(az_dates, t_lookback)
    start_data_idx = np.searchsorted(az_dates, t_start)

    if lookback_idx >= start_data_idx:
        continue
    if (az_dates[start_data_idx] - az_dates[lookback_idx]) < stability_td:
        continue
    if np.any(np.diff(az_dates[lookback_idx : start_data_idx + 1]) / np.timedelta64(1, "h") > 1.0):
        continue  # Data gap check
    if np.any(np.isnan(az_full[lookback_idx : start_data_idx + 1])):
        continue

    # Lookback must not contain points in the destination buffer
    window_segs_pre = full_segment_indices[lookback_idx : start_data_idx + 1]
    if np.any(window_segs_pre == dest_seg):
        continue

    # 2. Check Guard Window AFTER Transition (Lookforward)
    t_lookforward = t_end + stability_td
    end_data_idx = np.searchsorted(az_dates, t_end)
    lookforward_idx = np.searchsorted(az_dates, t_lookforward, side="right") - 1

    if lookforward_idx <= end_data_idx:
        continue
    if (az_dates[lookforward_idx] - az_dates[end_data_idx]) < stability_td:
        continue
    if np.any(np.diff(az_dates[end_data_idx : lookforward_idx + 1]) / np.timedelta64(1, "h") > 1.0):
        continue  # Data gap check
    if np.any(np.isnan(az_full[end_data_idx : lookforward_idx + 1])):
        continue

    # Lookforward must not contain points in the origin buffer
    window_segs_post = full_segment_indices[end_data_idx : lookforward_idx + 1]
    if np.any(window_segs_post == origin_seg):
        continue

    signal_validated_candidates.append(candidate)


# --- Step 4: Proximity Filtering (Mutual Destruction) ---
combined_events = []
num_candidates = len(signal_validated_candidates)
keep_mask = np.ones(num_candidates, dtype=bool)

# Disqualify neighboring transitions within stability threshold of each other
for i in range(num_candidates - 1):
    t_end_current = signal_validated_candidates[i]["t_end"]
    t_start_next = signal_validated_candidates[i + 1]["t_start"]

    if (t_start_next - t_end_current) < stability_td:
        keep_mask[i] = False
        keep_mask[i + 1] = False

# Compile verified events
for i, candidate in enumerate(signal_validated_candidates):
    if not keep_mask[i]:
        continue

    t_start = candidate["t_start"]
    t_end = candidate["t_end"]

    duration_days = (t_end - t_start) / np.timedelta64(1, "D")

    combined_events.append(
        {
            "duration_days": duration_days,
            "year": pd.Timestamp(t_start).year,
            "polarity": candidate["polarity"],
            "date": pd.Timestamp(t_start).to_pydatetime(),
        }
    )

combined_events.sort(key=lambda x: x["date"])

# Output copies
blocks_az = copy.deepcopy(blocks)
combined_events_az = copy.deepcopy(combined_events)

In [44]:
# =============================================================================
# PAD TRANSITION DETECTION & GUARD-WINDOW FILTERING
# =============================================================================

# Smooth values and extract dates safely from df_final
vals = gaussian_smooth_with_nans(df_pad["asym_255"].values, smooth)

# Extract date values safely across both Single and MultiIndex structures
if isinstance(df_pad.index, pd.MultiIndex):
    time_level = next(
        (name for name in df_pad.index.names if name and "time" in name.lower()), 0
    )
    time_values = df_pad.index.get_level_values(time_level)
else:
    time_values = df_pad.index

dates = pd.to_datetime(time_values).to_numpy(dtype="datetime64[ns]")

# Precompute time deltas in hours for instant gap detection
time_diffs_hours = np.diff(dates) / np.timedelta64(1, "h")

outval = 0.2
inval = 0.15

# --- Step 1: Vectorized Search for Transition Triggers ---
cross_up_idx = np.where((vals[1:] > -outval) & (vals[:-1] <= -outval))[0] + 1
cross_down_idx = np.where((vals[1:] < outval) & (vals[:-1] >= outval))[0] + 1

triggers = [(idx, "up") for idx in cross_up_idx] + [(idx, "down") for idx in cross_down_idx]
triggers.sort(key=lambda x: x[0])

candidate_blocks = []
last_end_idx = -1


# --- Step 2: Fast Scan Using NumPy Slice Searches ---
for start_idx, direction in triggers:
    if start_idx <= last_end_idx:
        continue

    remaining_vals = vals[start_idx:]
    if len(remaining_vals) == 0:
        continue

    nan_indices = np.where(np.isnan(remaining_vals))[0]
    first_nan = nan_indices[0] if len(nan_indices) > 0 else len(remaining_vals)

    if direction == "up":
        success_indices = np.where(remaining_vals >= outval)[0]
        if len(success_indices) == 0:
            continue
        first_success = success_indices[0]
        if first_nan < first_success:
            continue

        sub_vals = remaining_vals[: first_success + 1]
        if np.any(sub_vals <= -outval):
            continue

        above_minus_02 = np.where(sub_vals > -inval)[0]
        if len(above_minus_02) > 0:
            if np.any(sub_vals[above_minus_02[0] :] <= -inval):
                continue

        end_idx = start_idx + first_success
        candidate_blocks.append((start_idx - 1, end_idx, "up"))
        last_end_idx = end_idx

    else:  # direction == 'down'
        success_indices = np.where(remaining_vals <= -outval)[0]
        if len(success_indices) == 0:
            continue
        first_success = success_indices[0]
        if first_nan < first_success:
            continue

        sub_vals = remaining_vals[: first_success + 1]
        if np.any(sub_vals >= outval):
            continue

        below_02 = np.where(sub_vals < inval)[0]
        if len(below_02) > 0:
            if np.any(sub_vals[below_02[0] :] >= inval):
                continue

        end_idx = start_idx + first_success
        candidate_blocks.append((start_idx - 1, end_idx, "down"))
        last_end_idx = end_idx


# --- Step 3: NaN-Tolerant Guard Windows ---
signal_validated_blocks = []
stability_td = np.timedelta64(int(stability_hours), "h")

guard_stable_ratio = 0.75
max_nan_ratio = 0.20

for start_idx, end_idx, direction in candidate_blocks:
    t_start = dates[start_idx]
    t_end = dates[end_idx]

    # 1. Lookback Window
    t_lookback = t_start - stability_td
    lookback_idx = np.searchsorted(dates, t_lookback)
    pre_end = max(lookback_idx, start_idx - 1)

    if lookback_idx >= pre_end or (dates[start_idx] - dates[lookback_idx]) < stability_td:
        continue

    lookback_vals = vals[lookback_idx:pre_end]
    valid_pre = np.isfinite(lookback_vals)
    if np.mean(~valid_pre) > max_nan_ratio:
        continue

    clean_lookback = lookback_vals[valid_pre]
    if direction == "down" and np.mean(clean_lookback >= inval) < guard_stable_ratio:
        continue
    if direction == "up" and np.mean(clean_lookback <= -inval) < guard_stable_ratio:
        continue

    # 2. Lookforward Window
    t_lookforward = t_end + stability_td
    lookforward_idx = np.searchsorted(dates, t_lookforward, side="right") - 1
    post_start = min(lookforward_idx, end_idx + 1)

    if lookforward_idx <= post_start or (dates[lookforward_idx] - dates[end_idx]) < stability_td:
        continue

    lookforward_vals = vals[post_start : lookforward_idx + 1]
    valid_post = np.isfinite(lookforward_vals)
    if np.mean(~valid_post) > max_nan_ratio:
        continue

    clean_lookforward = lookforward_vals[valid_post]
    if direction == "down" and np.mean(clean_lookforward <= -inval) < guard_stable_ratio:
        continue
    if direction == "up" and np.mean(clean_lookforward >= inval) < guard_stable_ratio:
        continue

    signal_validated_blocks.append((start_idx, end_idx, direction))


# --- Step 4: Simultaneous Block-to-Block Proximity Isolation ---
combined_events_pad = []
num_blocks = len(signal_validated_blocks)

# Mask tracking block survival (True initially)
keep_mask = np.ones(num_blocks, dtype=bool)

# Simultaneous look-ahead/look-behind evaluation
for i in range(num_blocks - 1):
    t_end_current = dates[signal_validated_blocks[i][1]]
    t_start_next = dates[signal_validated_blocks[i + 1][0]]

    # Gap check against stability requirement
    if (t_start_next - t_end_current) < stability_td:
        keep_mask[i] = False
        keep_mask[i + 1] = False

# Compile verified events
for i, (start_idx, end_idx, direction) in enumerate(signal_validated_blocks):
    if not keep_mask[i]:
        continue  # Skip blocks knocked out by close neighbors

    t_start = dates[start_idx]
    t_end = dates[end_idx]

    duration_days = (t_end - t_start) / np.timedelta64(1, "D")
    polarity_tuple = (2, 1) if direction == "up" else (1, 2)

    combined_events_pad.append(
        {
            "duration_days": duration_days,
            "year": pd.Timestamp(t_start).year,
            "polarity": polarity_tuple,
            "date": pd.Timestamp(t_start).to_pydatetime(),
        }
    )

In [55]:
# =============================================================================
# EVENT CROSS-MATCHING & MERGING (PAD vs. AZIMUTH)
# =============================================================================

# Define the Polarity Translation Map (Matching)
pad_to_az_polarity = {
    (1, 2): (3, 1),  # PAD Up matches AZ (1 to 3)
    (2, 1): (1, 3),  # PAD Down matches AZ (3 to 1)
}

# Define Tuple-to-String Conversion Map
tuple_to_str_polarity = {
    (1, 2): "+,-",  # PAD equivalent
    (3, 1): "+,-",  # Matched AZ equivalent
    (2, 1): "-,+",  # PAD equivalent
    (1, 3): "-,+",  # Matched AZ equivalent
}

start_boundary = pd.to_datetime(start).to_pydatetime().replace(tzinfo=None)
match_window = timedelta(days=2.5)  # Buffer window applied to intervals

final_events = []
matched_az_indices = set()


# --- 1a. Process PAD events and search for mapped AZ matches ---
for pad_ev in combined_events_pad:
    if pad_ev["date"] < start_boundary:
        continue

    match_found = False
    expected_az_polarity = pad_to_az_polarity.get(pad_ev["polarity"])

    # Calculate PAD interval with +/- match_window buffer applied
    pad_start_buffered = pad_ev["date"] - match_window
    pad_end_buffered = (
        pad_ev["date"] + timedelta(days=pad_ev["duration_days"]) + match_window
    )

    if expected_az_polarity is not None:
        for j, az_ev in enumerate(combined_events_az):
            if j in matched_az_indices:
                continue

            if az_ev["polarity"] == expected_az_polarity:
                # Calculate AZ interval endpoints
                az_start = az_ev["date"]
                az_end = az_start + timedelta(days=az_ev["duration_days"])

                # Check if AZ interval overlaps with buffered PAD interval
                if pad_start_buffered <= az_end and pad_end_buffered >= az_start:
                    match_found = True
                    matched_az_indices.add(j)
                    break  # Match claimed

    # Build final event entry using PAD metadata
    merged_ev = pad_ev.copy()
    merged_ev["health"] = 1 if match_found else 2

    # Map tuple polarity to human-readable string representation
    merged_ev["polarity"] = tuple_to_str_polarity.get(
        pad_ev["polarity"], str(pad_ev["polarity"])
    )

    final_events.append(merged_ev)


# --- 1b. Append remaining unmatched AZ events ---
for j, az_ev in enumerate(combined_events_az):
    if j not in matched_az_indices:
        if az_ev["date"] < start_boundary:
            continue

        unmatched_ev = az_ev.copy()
        unmatched_ev["health"] = 3

        # Map tuple polarity to human-readable string representation
        unmatched_ev["polarity"] = tuple_to_str_polarity.get(
            az_ev["polarity"], str(az_ev["polarity"])
        )

        final_events.append(unmatched_ev)

# Sort merged event catalog chronologically
final_events.sort(key=lambda x: x["date"])

In [57]:
# =============================================================================
# CME & EVENT CROSS-MATCHING / TEMPORAL ASSOCIATION
# =============================================================================

# Maximum separation tolerance in hours to consider a CME associated
CME_TOLERANCE_HOURS = 48.0

# Evaluate CME overlap and temporal proximity for each identified event
for ev in final_events:
    ev_start = ev["date"]
    ev_end = ev_start + timedelta(days=ev["duration_days"])

    # Initialize default event metadata
    ev["cme"] = 0
    ev["cmepos"] = None
    ev["cmesep"] = 0.0

    best_cme_sep = float("inf")
    best_cme_pos = None

    for c_start, c_end in zip(rc_start, rc_end):
        # Ensure CME datetimes are offset-naive to match ev_start/ev_end
        c_start = c_start.replace(tzinfo=None) if getattr(c_start, "tzinfo", None) else c_start
        c_end = c_end.replace(tzinfo=None) if getattr(c_end, "tzinfo", None) else c_end

        # --- Determine Temporal Relationship & Separation (in hours) ---
        if c_end <= ev_start:
            # CME occurs before event
            pos = "b"
            sep_hours = (ev_start - c_end).total_seconds() / 3600.0

        elif c_start >= ev_end:
            # CME occurs after event
            pos = "a"
            sep_hours = (c_start - ev_end).total_seconds() / 3600.0

        else:
            # Event and CME overlap temporally
            sep_hours = 0.0

            if c_start <= ev_start and c_end >= ev_end:
                pos = "bao"  # Event fully contained inside CME duration
            elif c_start >= ev_start and c_end <= ev_end:
                pos = "o"    # CME fully contained inside event duration
            elif c_start < ev_start:
                pos = "bo"   # Overlaps, but CME starts before event
            elif c_end > ev_end:
                pos = "ao"   # Overlaps, but CME ends after event

        # --- Evaluate Proximity Window ---
        if sep_hours <= CME_TOLERANCE_HOURS:
            # Retain the closest CME if multiple fall within the window
            if sep_hours < best_cme_sep:
                best_cme_sep = sep_hours
                best_cme_pos = pos

    # --- Assign Nearest CME Results to Event ---
    if best_cme_pos is not None:
        ev["cme"] = 1
        ev["cmepos"] = best_cme_pos
        ev["cmesep"] = best_cme_sep if best_cme_pos in ["b", "a"] else 0.0

## Make plots

In [94]:


def save_individual_event_plots_to_png(
    df_omni,
    df_pad,
    final_events,
    rc_start,
    rc_end,
    output_dir,
):
    """Saves individual event plots as PNGs."""


    plt.rcParams.update({
        'font.size': 14,          # Base text size
        'axes.labelsize': 15,     # Axis X and Y labels
        'axes.titlesize': 16,     # Subplot titles
        'xtick.labelsize': 13,    # X-axis tick labels
        'ytick.labelsize': 13,    # Y-axis tick labels
        'legend.fontsize': 14,    # Legend font size
        'figure.titlesize': 18    # Main figure title
    })
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    utc_dt_format1 = DateFormatter("%m.%d %H:%M")

    # --- Pre-process OMNI Data ---
    df_working = df_omni.copy()
    if "DATETIME" in df_working.columns:
        df_working["DATETIME"] = pd.to_datetime(df_working["DATETIME"])
        df_working = df_working.set_index("DATETIME")
    else:
        df_working.index = pd.to_datetime(df_working.index)

    df_working = df_working.sort_index()

    # Calculate Azimuth
    az_full, az_dates = process_and_smooth_azimuth(
        df_working["BX_GSE1800"].values,
        df_working["BY_GSE1800"].values,
        df_working.index.values,
        sigma=6,
    )

    # Extract optional Beta and Temperature columns with fallbacks
    beta_col = next((c for c in ["Beta1800", "BETA", "Beta"] if c in df_working.columns), None)
    temp_col = next((c for c in ["T1800", "TEMP", "Temperature"] if c in df_working.columns), None)

    df = pd.DataFrame(
        {
            "BX": df_working["BX_GSE1800"],
            "BY": df_working["BY_GSE1800"],
            "BZ": df_working["BZ_GSE1800"],
            "V": np.where(
                df_working["V1800"] < 2000, df_working["V1800"], np.nan
            ),
            "N": df_working["N1800"],
            "AZ": az_full,
            "BETA": df_working[beta_col] if beta_col else np.nan,
            "TEMP": df_working[temp_col] if temp_col else np.nan,
        },
        index=df_working.index,
    )

    # --- Pre-process PAD Data & Extract Pitch Angles ---
    if not isinstance(df_pad.index, pd.DatetimeIndex):
        df_pad.index = pd.to_datetime(df_pad.index)

    time_axis = df_pad.index.values

    pa_cols = [
        col
        for col in df_pad.columns
        if str(col).startswith("pa_")
        or col.replace(".", "", 1).replace("-", "", 1).isdigit()
    ]

    if hasattr(df_pad, "attrs") and "pitch_angles" in df_pad.attrs:
        wind_pa = np.asarray(df_pad.attrs["pitch_angles"]).flatten()
    else:
        wind_pa = np.linspace(0, 180, len(pa_cols))

    wind_e255 = df_pad[pa_cols].values.T
    wind_e255_normalized = normalised_pad(wind_e255)

    insitu = getattr(df_omni, "attrs", {}).get("insitu", [])
    buffer_td = timedelta(days=8)

    # --- Iterate through each event to render PNG ---
    for target_idx, target_ev in enumerate(final_events):
        ev_start = pd.to_datetime(target_ev["date"])
        ev_end = ev_start + timedelta(days=float(target_ev["duration_days"]))

        plot_start = ev_start - buffer_td
        plot_end = ev_end + buffer_td

        mask = (df.index >= plot_start) & (df.index < plot_end)
        page_df = df.loc[mask]

        if page_df.empty:
            continue

        fig, (ax0, ax1, ax2, ax3, ax4) = plt.subplots(
            5, 1, figsize=(15.69, 15.0), sharex=True
        )
        plt.subplots_adjust(
            hspace=0.1, top=0.92, bottom=0.10, left=0.1, right=0.88
        )

        # --- PANEL 0: V-N Components ---
        ax0.set_xlim(plot_start, plot_end)
        ln1 = ax0.plot(
            page_df.index,
            page_df["V"],
            color="tab:blue",
            lw=1,
            label="Velocity",
            alpha=0.7,
        )
        ax0.set_ylabel(r"Velocity $v$ [km$~$s$^{-1}$]")

        ax0_right = ax0.twinx()
        ln2 = ax0_right.plot(
            page_df.index,
            page_df["N"],
            color="tab:orange",
            lw=1,
            label="Density",
            alpha=0.7,
        )
        ax0_right.set_ylabel(r"Density n [cm$^{-3}$]")

        lines = ln1 + ln2
        labels = [l.get_label() for l in lines]
        ax0.legend(
            lines, labels, loc="upper right", fontsize="x-small", ncol=2
        )
        ax0.set_ylim(300, np.max(np.array([np.nanmax(page_df["V"])*1.2,750])))
        ax0_right.set_ylim(0, np.nanmax(page_df["N"])*1.2 )

        # --- PANEL 1: B-Field Components ---
        ax1.set_xlim(plot_start, plot_end)
        b1 = ax1.plot(
            page_df.index,
            page_df["BX"],
            color="#d62728",
            lw=1.2,
            label=r"B$_x^{\mathrm{GSE}}$",
            alpha=0.7,
        )[0]
        b2 = ax1.plot(
            page_df.index,
            page_df["BY"],
            color="#2ca02c",
            lw=1.2,
            label=r"B$_y^{\mathrm{GSE}}$",
            alpha=0.7,
        )[0]
        b3 = ax1.plot(
            page_df.index,
            page_df["BZ"],
            color="#1f77b4",
            lw=1.2,
            label=r"B$_z^{\mathrm{GSE}}$",
            alpha=0.7,
        )[0]

        ax1.set_ylabel("B [nT]")
        valmax=np.max(np.array([np.nanmax(np.abs(ci)) for ci in [page_df["BX"],page_df["BY"],page_df["BZ"]]]))
        ax1.set_ylim(-valmax*1.2,valmax*1.2,)
        ax1.axhline(0.0, color="gray", lw=1, ls="--", alpha=0.5)

        lines_b = [b1, b2, b3]
        labels_b = [l.get_label() for l in lines_b]
        first_legend = ax1.legend(
            lines_b, labels_b, loc="upper right", fontsize="x-small", ncol=3
        )
        ax1.add_artist(first_legend)

        # --- PANEL 2 (3rd position): Plasma Beta & Temperature ---
        ax2.set_xlim(plot_start, plot_end)
        l_beta = ax2.plot(
            page_df.index,
            page_df["BETA"],
            color="purple",
            lw=1.5,
            label="Plasma Beta",
            alpha=0.8,
        )
        ax2.axhline(1.0, color="gray", lw=1, ls="--", alpha=0.5)
        ax2.set_ylim(np.nanmin(page_df["BETA"])*0.1,np.nanmax(page_df["BETA"])*10)
        ax2.set_ylabel("Plasma Beta")#, color="purple")
        ax2.set_yscale("log")

        ax2_right = ax2.twinx()
        l_temp = ax2_right.plot(
            page_df.index,
            page_df["TEMP"],
            color="red",
            lw=1.5,
            label="Temperature",
            alpha=0.7,
        )
        ax2_right.set_ylabel("Temperature [K]")#, color="red")
        ax2_right.set_ylim(0,np.nanmax(page_df["TEMP"])*1.20)
        
        lines_bt = l_beta + l_temp
        labels_bt = [l.get_label() for l in lines_bt]
        ax2.legend(
            lines_bt, labels_bt, loc="upper right", fontsize="x-small", ncol=2
        )

        # --- PANEL 3: Azimuth ---
        res1 = ax3.plot(
            page_df.index,
            page_df["AZ"],
            color="firebrick",
            lw=1,
            label="Azimuth",
        )[0]
        ax3.set_ylabel("Azimuth [rad]")
        ax3.set_ylim(-np.pi, np.pi)
        ax3.set_yticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
        ax3.set_yticklabels(
            [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"]
        )
        for y_val in [-np.pi / 2, 0, np.pi / 2]:
            ax3.axhline(y_val, color="gray", lw=0.5, ls="--", alpha=0.5)

        ax3_right = ax3.twinx()
        if "asym_255" in df_pad.columns:
            mask2 = (df_pad.index >= plot_start) & (df_pad.index < plot_end)
            page_df2 = df_pad.index[mask2]

            res2 = ax3_right.plot(
                page_df2,
                gaussian_smooth_with_nans(df_pad["asym_255"][mask2], 6),
                color="darkslateblue",
                linewidth=1.2,
                alpha=0.8,
                label="Asymmetry Index at 255 eV",
            )[0]

            lines_az = [res1, res2]
            labels_az = [l.get_label() for l in lines_az]
            ax3.legend(
                lines_az,
                labels_az,
                loc="upper right",
                fontsize="x-small",
                ncol=2,
            )

        ax3_right.set_ylim(1.0, -1.0)
        ax3_right.set_ylabel("Asymmetry Index", color="black")
        ax3_right.axhline(0.25, color="gray", lw=0.5, ls="--", alpha=0.5)
        ax3_right.axhline(-0.25, color="gray", lw=0.5, ls="--", alpha=0.5)
        ax3_right.axhline(0.0, color="gray", lw=0.5, ls="-", alpha=0.5)

        # --- PANEL 4: Electron Asymmetry & PAD ---
        wind_mask = (time_axis >= plot_start) & (time_axis < plot_end)
        sliced_time = time_axis[wind_mask]
        sliced_e255_norm = wind_e255_normalized[:, wind_mask]

        if sliced_time.size > 0:
            T, PA = np.meshgrid(mdates.date2num(sliced_time), wind_pa)

            pcm4 = ax4.pcolormesh(
                T,
                PA,
                sliced_e255_norm,
                norm=colors.LogNorm(vmin=0.2, vmax=10),
                cmap="jet",
                shading="nearest",
                rasterized=True,
            )
            pcm4.set_array(sliced_e255_norm)

            pos4 = ax4.get_position()
            cax_position = [pos4.x1 + 0.02, pos4.y0, 0.015, pos4.height]
            cax4 = fig.add_axes(cax_position)

            cbar4 = fig.colorbar(pcm4, cax=cax4)
            cbar4.ax.set_ylabel(
                "Normalised electron PAD\n"
                + r"at energy 255 eV [s$^3$/km$^6$]",
                rotation=90,
                labelpad=16,
            )

        ax4.set_ylabel("Pitch Angle [deg]")

        # --- OVERLAYS ACROSS ALL PANELS ---
        all_axes = [ax0, ax1, ax2, ax3, ax4]
        for ax in all_axes:
            # Svalgaard In-situ lines
            for ev in insitu:
                ev_ts = pd.to_datetime(ev["timestamp"])
                if plot_start <= ev_ts < plot_end:
                    line_c = (
                        "darkred"
                        if ev.get("polarity") == "-,+"
                        else "darkblue"
                    )
                    ax.axvline(
                        ev_ts,
                        color=line_c,
                        linestyle="--",
                        lw=1.5,
                        alpha=0.8,
                    )

            # Event Boundaries Top/Bottom Bar Overlays
            for idx, cev in enumerate(final_events):
                c_start = pd.to_datetime(cev["date"])
                c_end = c_start + timedelta(days=float(cev["duration_days"]))

                if c_start < plot_end and c_end > plot_start:
                    fill_start = max(c_start, plot_start)
                    fill_end = min(c_end, plot_end)
                    duration_sec = (c_end - c_start).total_seconds()

                    pol = cev.get("polarity", "")
                    if pol == "+,-":
                        fill_c = "#0B1D3A"
                    elif pol == "-,+":
                        fill_c = "#FF4500"
                    else:
                        fill_c = "#708090"

                    is_target = idx == target_idx

                    if duration_sec > 0:
                        ax.axvspan(
                            fill_start,
                            fill_end,
                            color=fill_c,
                            alpha=1.0 if is_target else 0.5,
                            ymin=0.95,
                            ymax=0.99,
                            zorder=5,
                        )
                        ax.axvspan(
                            fill_start,
                            fill_end,
                            color=fill_c,
                            alpha=1.0 if is_target else 0.5,
                            ymin=0.01,
                            ymax=0.05,
                            zorder=5,
                        )
                        ax.axvline(
                            fill_start,
                            color=fill_c,
                            linestyle=":",
                            lw=1.5,
                            alpha=0.8,
                            zorder=5,
                        )
                        ax.axvline(
                            fill_end,
                            color=fill_c,
                            linestyle=":",
                            lw=1.5,
                            alpha=0.8,
                            zorder=5,
                        )
                        ax.axvspan(
                            fill_start,
                            fill_end,
                            color=fill_c,
                            alpha=0.1,
                            ymin=0,
                            ymax=1,
                            zorder=100,
                        )
                        
                    else:
                        ax.axvline(
                            fill_start,
                            color=fill_c,
                            lw=2,
                            alpha=1.0,
                            zorder=5,
                        )

                    if is_target and ax == ax0:
                        mid_point = fill_start + (fill_end - fill_start) / 2

            # CME Boundaries
            for ev_start_cme, ev_end_cme in zip(rc_start, rc_end):
                ev_start_cme = pd.to_datetime(ev_start_cme).replace(tzinfo=None)
                ev_end_cme = pd.to_datetime(ev_end_cme).replace(tzinfo=None)

                if ev_start_cme < plot_end and ev_end_cme > plot_start:
                    fill_start_cme = max(ev_start_cme, plot_start)
                    fill_end_cme = min(ev_end_cme, plot_end)
                    duration_cme_sec = (
                        ev_end_cme - ev_start_cme
                    ).total_seconds()

                    if duration_cme_sec > 0:
                        ax.axvspan(
                            fill_start_cme,
                            fill_end_cme,
                            facecolor="none",
                            edgecolor="tab:orange",
                            linestyle=":",
                            linewidth=2,
                            ymin=0.05,
                            ymax=0.95,
                            zorder=2,
                        )
                    else:
                        ax.axvline(
                            fill_start_cme,
                            color="tab:orange",
                            lw=2,
                            ls=":",
                        )

        # Legend Handles
        p1_patch = mpatches.Patch(
            color="#0B1D3A", alpha=1, label="Polarity (in,out)"
        )
        p2_patch = mpatches.Patch(
            color="#FF4500", alpha=1, label="Polarity (out,in)"
        )
        darkred_handle = mlines.Line2D(
                    [], [], 
                    color='orange', 
                    linestyle=':', 
                    linewidth=2, 
                    alpha=0.8, 
                    label='CME (R&C List)'  # e.g., "Polarity (+,-)"
                )
        
        ax0.legend(
            handles=[p1_patch, p2_patch,darkred_handle],
            loc="lower left",
            fontsize="small",
            ncol=4,
            bbox_to_anchor=(0.0, 0.97),
            frameon=False,
        )

        # Frame and Ticks Formatting
        twin_axes = [ax0_right, ax2_right, ax3_right]
        for ax in all_axes + twin_axes:
            ax.tick_params(axis="both", which="major", length=10, width=2)
            ax.tick_params(axis="both", which="minor", length=5, width=0.5)
            ax.minorticks_on()
            for spine in ax.spines.values():
                spine.set_linewidth(2)

        fig.suptitle(
            f"Date: {ev_start.strftime('%Y-%m-%d %H:%M:%S')}",# | Polarity: {target_ev.get('polarity', 'N/A')} ",
            y=0.96,
            weight="bold",
        )

        ax4.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax4.tick_params(axis="x", labelrotation=20)
        ax4.set_xlabel("Date")

        for ax in all_axes:
            ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
            ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))
            ax.grid(
                which="major",
                axis="x",
                linestyle=":",
                linewidth=0.5,
                color="black",
            )
            ax.grid(
                which="minor",
                axis="x",
                linestyle=":",
                linewidth=0.5,
                color="gray",
                alpha=0.7,
            )

        filename = f"{ev_start.strftime('%Y%m%d_%H%M%S')}.png"
        save_path = os.path.join(output_dir, filename)

        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print(f"All event plots saved successfully to {output_dir}")

# --- Updated Call ---
save_individual_event_plots_to_png(
    df_omni,
    df_pad,
    final_events,
    rc_start,
    rc_end,
    path+"/png_events/"
)

/var/folders/bq/gbxqd31127n61zblp_qxplbh0000gn/T/ipykernel_1796/1023381616.py:121: RuntimeWarning: Mean of empty slice
  energy_channel_means = np.nanmean(energy_channel, axis=0) #column means


All event plots saved successfully to /Users/sgh/Documents/Projects/HCS_statistics/plots/png_events/


## compute SBC properties

In [98]:
def calculate_parker_thickness(omni_df, event):
    """Slices df_omni for a specific event and calculates HPS thickness

    using a dynamically calculated Parker spiral angle based on observed bulk
    velocity.
    """
    # 1. Calculate start and end times for the event slice
    start_time = pd.to_datetime(event["date"])
    duration_days = float(event["duration_days"])
    end_time = start_time + timedelta(days=duration_days)

    # 2. Slice the main dataframe using time mask
    time_mask = (omni_df.index >= start_time) & (omni_df.index <= end_time)
    event_slice = omni_df[time_mask]

    if event_slice.empty:
        print(f"Skipped event starting at {start_time}: Data slice is empty.")
        return None

    # 3. Flexible velocity column detection (supports V1800, V, v)
    v_col = next(
        (c for c in ["V1800", "V", "v", "V_GSE"] if c in omni_df.columns), None
    )

    if v_col is None:
        print(
            f"Skipped event starting at {start_time}: Velocity column not found in DataFrame."
        )
        return None

    # Clean missing/fill values
    v_data = event_slice[v_col].dropna()
    v_data = v_data[v_data < 2000]

    if len(v_data) == 0:
        print(
            f"Skipped event starting at {start_time}: Missing valid velocity data."
        )
        return None

    # Average radial speed (Vr) over the crossing duration
    v_radial = v_data.mean()

    # 4. Total duration in seconds derived directly from event duration
    delta_t_sec = duration_days * 86400.0

    # 5. Dynamic Parker spiral angle calculation (Omega * r at 1 AU ~ 404 km/s)
    omega_r_1au = 404.0
    angle_rad = np.arctan(omega_r_1au / v_radial)
    parker_angle_deg = np.degrees(angle_rad)

    # 6. Apply Parker geometry: V_normal = V_radial * cos(spiral_angle)
    v_normal = v_radial * np.cos(angle_rad)

    # 7. Physical thickness: L = V_normal * delta_t
    thickness_km = v_normal * delta_t_sec

    return {
        "date": start_time,
        "thickness_km": thickness_km,
        "v_radial_kms": v_radial,
        "v_normal_kms": v_normal,
        "parker_angle_deg": parker_angle_deg,
        "duration_sec": delta_t_sec,
    }


# --- Prepare df_omni Index & Run Loop ---

# Ensure df_omni index is set to DatetimeIndex using existing 'Time' or 'DATETIME' column
if "Time" in df_omni.columns:
    df_omni["Time"] = pd.to_datetime(df_omni["Time"])
    df_omni.set_index("Time", inplace=True)
elif "DATETIME" in df_omni.columns:
    df_omni["DATETIME"] = pd.to_datetime(df_omni["DATETIME"])
    df_omni.set_index("DATETIME", inplace=True)
elif not isinstance(df_omni.index, pd.DatetimeIndex):
    df_omni.index = pd.to_datetime(df_omni.index)

df_omni.sort_index(inplace=True)

results_list_parker = []

for event in final_events:
    res = calculate_parker_thickness(df_omni, event)

    if res is not None:
        results_list_parker.append(res)
        print(
            f"Processed Parker event {res['date']}: "
            f"Vr = {res['v_radial_kms']:.1f} km/s | "
            f"Angle = {res['parker_angle_deg']:.1f}° | "
            f"Thickness = {res['thickness_km']:,.1f} km"
        )
    else:
        results_list_parker.append(
            {
                "date": pd.to_datetime(event["date"]),
                "thickness_km": np.nan,
                "v_radial_kms": np.nan,
                "v_normal_kms": np.nan,
                "parker_angle_deg": np.nan,
                "duration_sec": np.nan,
            }
        )
        print(f"Skipped event starting at {event['date']} (logged as NaN).")

df_parker_results = pd.DataFrame(results_list_parker)

Processed Parker event 2023-01-04 04:30:00: Vr = 392.0 km/s | Angle = 45.9° | Thickness = 31,443,166.0 km
Processed Parker event 2023-01-16 07:30:00: Vr = 432.0 km/s | Angle = 43.1° | Thickness = 45,435,492.0 km
Processed Parker event 2023-01-29 19:30:00: Vr = 415.7 km/s | Angle = 44.2° | Thickness = 26,832,810.0 km
Processed Parker event 2023-02-13 03:30:00: Vr = 367.7 km/s | Angle = 47.7° | Thickness = 9,799,593.0 km
Skipped event starting at 2023-02-25 17:30:00: Missing valid velocity data.
Skipped event starting at 2023-02-25 17:30:00 (logged as NaN).
Processed Parker event 2023-03-13 18:30:00: Vr = 330.8 km/s | Angle = 50.7° | Thickness = 8,300,374.5 km
Processed Parker event 2023-03-24 10:30:00: Vr = 425.6 km/s | Angle = 43.5° | Thickness = 11,113,721.0 km


In [99]:

all_event_velocities, all_event_velocities_sig = [], []
all_event_densities, all_event_densities_sig, all_event_densities_peak = (
    [],
    [],
    [],
)
all_event_b_fields, all_event_b_fields_sig = [], []
all_event_betas, all_event_betas_sig = [], []

# Ensure df_omni has a proper DatetimeIndex
if "Time" in df_omni.columns:
    df_omni["Time"] = pd.to_datetime(df_omni["Time"])
    df_omni.set_index("Time", inplace=True)
elif "DATETIME" in df_omni.columns:
    df_omni["DATETIME"] = pd.to_datetime(df_omni["DATETIME"])
    df_omni.set_index("DATETIME", inplace=True)
elif not isinstance(df_omni.index, pd.DatetimeIndex):
    df_omni.index = pd.to_datetime(df_omni.index)

df_omni.sort_index(inplace=True)


# Dynamic column finder helper
def get_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


v_col = get_col(df_omni, ["V1800", "V", "v", "V_GSE"])
den_col = get_col(df_omni, ["N1800", "N", "DENSITY", "Density", "n"])
beta_col = get_col(df_omni, ["Beta1800", "BETA1800", "Beta", "BETA", "beta"])
bx_col = get_col(df_omni, ["BX_GSE1800", "BX_GSE", "Bx", "BX"])
by_col = get_col(df_omni, ["BY_GSE1800", "BY_GSE", "By", "BY"])
bz_col = get_col(df_omni, ["BZ_GSE1800", "BZ_GSE", "Bz", "BZ"])

for e in final_events:
    start_time = pd.to_datetime(e["date"])
    end_time = start_time + timedelta(days=float(e["duration_days"]))

    seg = df_omni.loc[(df_omni.index >= start_time) & (df_omni.index <= end_time)]

    if seg.empty:
        v_valid = den_valid = beta_valid = b_valid = np.array([])
    else:
        # Extract and validate Velocity
        if v_col:
            v_seg = pd.to_numeric(seg[v_col], errors="coerce")
            v_valid = v_seg[(v_seg < 2000) & (v_seg > 0)].values
        else:
            v_valid = np.array([])

        # Extract and validate Density
        if den_col:
            den_seg = pd.to_numeric(seg[den_col], errors="coerce")
            den_valid = den_seg[(den_seg < 500) & (den_seg > 0)].values
        else:
            den_valid = np.array([])

        # Extract and validate Plasma Beta
        if beta_col:
            beta_seg = pd.to_numeric(seg[beta_col], errors="coerce")
            beta_valid = beta_seg[(beta_seg < 1000) & (beta_seg >= 0)].values
        else:
            beta_valid = np.array([])

        # Extract B-Field components and compute magnitude
        if bx_col and by_col and bz_col:
            bx = pd.to_numeric(seg[bx_col], errors="coerce")
            by = pd.to_numeric(seg[by_col], errors="coerce")
            bz = pd.to_numeric(seg[bz_col], errors="coerce")
            b_seg = np.sqrt(bx**2 + by**2 + bz**2)
            b_valid = b_seg[(b_seg < 200) & (b_seg >= 0)].dropna().values
        else:
            b_valid = np.array([])

    # Velocity stats
    all_event_velocities.append(
        np.nanmean(v_valid) if len(v_valid) > 0 else np.nan
    )
    all_event_velocities_sig.append(
        np.nanstd(v_valid) if len(v_valid) > 0 else np.nan
    )

    # Density stats
    all_event_densities.append(
        np.nanmean(den_valid) if len(den_valid) > 0 else np.nan
    )
    all_event_densities_sig.append(
        np.nanstd(den_valid) if len(den_valid) > 0 else np.nan
    )
    all_event_densities_peak.append(
        np.nanmax(den_valid) if len(den_valid) > 0 else np.nan
    )

    # B-field stats
    all_event_b_fields.append(
        np.nanmean(b_valid) if len(b_valid) > 0 else np.nan
    )
    all_event_b_fields_sig.append(
        np.nanstd(b_valid) if len(b_valid) > 0 else np.nan
    )

    # Beta stats
    all_event_betas.append(
        np.nanmean(beta_valid) if len(beta_valid) > 0 else np.nan
    )
    all_event_betas_sig.append(
        np.nanstd(beta_valid) if len(beta_valid) > 0 else np.nan
    )

# Final array conversions
velocities = np.array(all_event_velocities)
sigs_v = np.array(all_event_velocities_sig)

densities = np.array(all_event_densities)
sigs_den = np.array(all_event_densities_sig)
peak_densities = np.array(all_event_densities_peak)

b_fields = np.array(all_event_b_fields)
sigs_b = np.array(all_event_b_fields_sig)

betas = np.array(all_event_betas)
sigs_beta = np.array(all_event_betas_sig)

## save to .csv file

In [103]:
csv_rows = []

for i, e in enumerate(final_events):
    # 1. Dates
    start_time = e['date']
    end_time = start_time + timedelta(days=float(e['duration_days']))

    if e['date'] != df_parker_results['date'][i]:
        print(e['date'] ,df_parker_results['date'][i])
        
    start_str = start_time.strftime('%Y-%m-%d %H:%M')
    end_str = end_time.strftime('%Y-%m-%d %H:%M')
    
    # 2. Polarity
    polarity_str = str(e['polarity'])
    
    # 3. Duration in hours
    duration_hours = float(e['duration_days']) * 24.0
    
    # Helper to clean/round float values for CSV output
    def clean_val(val, decimals=2):
        if val is None or np.isnan(val):
            return np.nan
        return round(float(val), decimals)

    # 4. Confidence Mapping (health)
    health_val = e.get('health')
    if health_val == 1:
        conf_str = "high"
    elif health_val == 2:
        conf_str = "moderate"
    elif health_val == 3:
        conf_str = "low"
    else:
        conf_str = "-"
        
    # 5. CME (y/n)
    cme_val = e.get('cme', 0)
    cme_str = "y" if cme_val in [1, True, '1', 'y', 'Y'] else "n"
    
    # 6. CME Position (b/a/o)
    cme_pos_val = str(e.get('cmepos')).lower().strip()
    if cme_pos_val == 'a':
        cme_pos_str = 'a'
    elif cme_pos_val == 'b':
        cme_pos_str = 'b'
    elif cme_pos_val in ['bao', 'bo', 'ao', 'o']:
        cme_pos_str = 'o'
    else:
        cme_pos_str = '-'

    # Append row record with explicit separate columns
    csv_rows.append({
        'Start Time': start_str,
        'End Time': end_str,
        'Polarity': polarity_str,
        'Duration (hours)': clean_val(duration_hours, 2),
        'Thickness (km)': clean_val(df_parker_results['thickness_km'][i], 2),
        
        # Velocity
        'Velocity Avg (km/s)': clean_val(velocities[i], 1),
        'Velocity Std (km/s)': clean_val(sigs_v[i], 1),
        
        # Density
        'Density Avg (cm^-3)': clean_val(densities[i], 2),
        'Density Std (cm^-3)': clean_val(sigs_den[i], 2),
        'Peak Density (cm^-3)': clean_val(peak_densities[i], 2),
        
        # Magnetic Field Magnitude
        'B Field Avg (nT)': clean_val(b_fields[i], 2),
        'B Field Std (nT)': clean_val(sigs_b[i], 2),
        
        # Plasma Beta
        'Beta Avg': clean_val(betas[i], 2),
        'Beta Std': clean_val(sigs_beta[i], 2),
        
        # Metadata
        'Confidence': conf_str,
        'CME': cme_str,
        'CME Position': cme_pos_str
    })

# Convert to DataFrame
df_events = pd.DataFrame(csv_rows)

# Write to CSV
filename = f"{'Results_'+
            pd.to_datetime(start).tz_localize(None).floor("h").strftime('%Y%m%d_%H%M%S')+'_'+
            pd.to_datetime(end).tz_localize(None).floor("h").strftime('%Y%m%d_%H%M%S')}.csv"
output_filename = os.path.join(path, filename)

df_events.to_csv(output_filename, index=False, encoding="utf-8")

print(f"CSV file successfully written to {output_filename}")

CSV file successfully written to /Users/sgh/Documents/Projects/HCS_statistics/Results_20230101_000000_20230410_000000.csv
